# Sistemas recomendadores de películas
Práctica de la lección: ranking ponderado, recomendaciones por sinopsis y recomendaciones por metadatos.

Ejecuta las celdas en orden con Python 3, pandas, numpy y scikit-learn. Coloca los CSV en `data/` junto a este notebook o en la raíz del repositorio. `credits.csv` es opcional: sin él se usan géneros y keywords, y se señala explícitamente la limitación.

Los datos describen películas hasta 2017. No representan el catálogo actual ni preferencias personales. TF-IDF representa frecuencias de términos, no embeddings semánticos aprendidos.

In [1]:
from pathlib import Path
from ast import literal_eval
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from IPython.display import display

def data_path(name, required=True):
    for folder in (Path('data'), Path('.'), Path('../data')):
        path = folder / name
        if path.is_file():
            return path
    if required:
        raise FileNotFoundError(f'Coloca {name} en data/ junto al notebook.')
    return None

df = pd.read_csv(data_path('movies_metadata.csv'), low_memory=False)
display(df.head(3))
print('Dimensiones:', df.shape)
display(df[['title', 'overview', 'genres', 'vote_average', 'vote_count']].isna().sum())

Dimensiones: (45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


title             6
overview        954
genres            0
vote_average      6
vote_count        6
dtype: int64

## 1. Recomendador simple
La puntuación es $WR = v/(v+m)R + m/(v+m)C$. `C` es la media global de las calificaciones y `m` el percentil 90 del número de votos. Este ranking ofrece las mismas sugerencias a todos los usuarios.

In [2]:
df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce')
df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce')
C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.90)
print(f'C = {C:.12f}; m = {m}')

def weighted_rating(x, m=m, C=C):
    v, R = x['vote_count'], x['vote_average']
    return v / (v + m) * R + m / (v + m) * C

q_movies = df.loc[df['vote_count'] >= m].copy()
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)
q_movies = q_movies.sort_values('score', ascending=False)
print('Películas elegibles:', len(q_movies))
top_250 = q_movies[['title', 'vote_count', 'vote_average', 'score']].head(250)
display(top_250.head(15))

C = 5.618207215134; m = 160.0
Películas elegibles: 4555


,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


## 2. Recomendador por sinopsis
Sustituimos las sinopsis faltantes por cadenas vacías y usamos stop words en inglés, idioma predominante del texto.

TF-IDF normaliza cada fila. Su producto punto equivale a la similitud coseno. Calculamos solo la fila consultada: una matriz completa de 45,466 × 45,466 requeriría aproximadamente 16.5 GB en float64. El cálculo por consulta conserva la métrica de la lección.

Los títulos repetidos se resuelven con `movie_id`; cuando no se proporciona, se pide desambiguar. Se excluye la película por índice y las similitudes cero.

In [3]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'].fillna(''))
print('Matriz TF-IDF:', tfidf_matrix.shape)
display(tfidf.get_feature_names_out()[5000:5010])

def get_recommendations(title, matrix=tfidf_matrix, catalog=df, n=10,
                        movie_id=None, tfidf_normalized=True):
    if not isinstance(n, int) or n < 1:
        raise ValueError('n debe ser un entero positivo.')
    matches = np.flatnonzero(catalog['title'].fillna('').eq(title).to_numpy())
    if movie_id is not None:
        ids = pd.to_numeric(catalog['id'], errors='coerce').to_numpy()
        matches = matches[ids[matches] == int(movie_id)]
    if len(matches) == 0:
        raise ValueError(f'Película no encontrada: {title!r}.')
    if len(matches) > 1:
        options = catalog.iloc[matches][['id', 'title', 'release_date']]
        raise ValueError(f'Título ambiguo; indica movie_id:\n{options.to_string(index=False)}')
    idx = int(matches[0])
    metric = linear_kernel if tfidf_normalized else cosine_similarity
    scores = metric(matrix[idx], matrix).ravel()
    scores[idx] = -1
    positions = np.argsort(-scores, kind='stable')
    positions = positions[scores[positions] > 0][:n]
    result = catalog.iloc[positions][['id', 'title', 'release_date']].copy()
    result['similarity'] = scores[positions]
    return result.reset_index(drop=True)

display(get_recommendations('The Dark Knight Rises'))
display(get_recommendations('The Godfather'))

Matriz TF-IDF: (45466, 75827)


array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

,id,title,release_date,similarity
0,155,The Dark Knight,2008-07-16,0.326639
1,414,Batman Forever,1995-06-16,0.316211
2,364,Batman Returns,1992-06-19,0.296139
3,40662,Batman: Under the Red Hood,2010-07-27,0.287983
4,268,Batman,1989-06-23,0.270730
5,29751,Batman Unmasked: The Psychology of the Dark Kn...,2008-07-15,0.253149
6,16234,Batman Beyond: Return of the Joker,2000-12-12,0.234360
7,69735,Batman: Year One,2011-09-27,0.204338
8,123025,"Batman: The Dark Knight Returns, Part 1",2012-09-06,0.198813
9,14919,Batman: Mask of the Phantasm,1993-12-25,0.188744


,id,title,release_date,similarity
0,240,The Godfather: Part II,1974-12-20,0.475954
1,364150,The Godfather Trilogy: 1972-1990,1992-10-17,0.356142
2,242,The Godfather: Part III,1990-12-24,0.183049
3,190955,Blood Ties,2013-08-22,0.153702
4,119907,Household Saints,1993-09-15,0.140767
5,202239,Start Liquidation,1983-03-12,0.134386
6,18747,Election,2005-05-14,0.132326
7,135335,A Mother Should Be Loved,1934-05-11,0.130311
8,407,Short Sharp Shock,1998-10-15,0.129697
9,315362,Beck 28 - Familjen,2015-01-10,0.127998


## 3. Recomendador por metadatos
Convertimos listas serializadas con `literal_eval`, limpiamos IDs inválidos por su contenido y evitamos multiplicar filas por IDs repetidos en los CSV adicionales.

La lección usa los tres actores principales, director, géneros y keywords. Los archivos recibidos no incluyen `credits.csv`: si está disponible se incorpora automáticamente; en caso contrario esta ejecución utiliza únicamente keywords y géneros. No se inventan actores ni directores.

In [4]:
def parse_list(value):
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return []
    try:
        parsed = literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []

def valid_ids(frame):
    frame = frame.copy()
    ids = pd.to_numeric(frame['id'], errors='coerce')
    frame = frame.loc[ids.notna() & ids.mod(1).eq(0)].copy()
    frame['id'] = ids.loc[frame.index].astype('int64')
    return frame

metadata = valid_ids(df).drop_duplicates('id').reset_index(drop=True)
keywords = valid_ids(pd.read_csv(data_path('keywords.csv'))).drop_duplicates('id')
metadata = metadata.merge(keywords[['id', 'keywords']], on='id', how='left', validate='one_to_one')
credits_path = data_path('credits.csv', required=False)
has_credits = credits_path is not None
if has_credits:
    credits = valid_ids(pd.read_csv(credits_path)).drop_duplicates('id')
    metadata = metadata.merge(credits[['id', 'cast', 'crew']], on='id', how='left', validate='one_to_one')
    print('Modelo completo: actores, director, keywords y géneros.')
else:
    metadata['cast'] = [[] for _ in range(len(metadata))]
    metadata['crew'] = [[] for _ in range(len(metadata))]
    print('Modelo parcial: keywords y géneros; falta credits.csv para actores y director.')

def get_director(crew):
    return next((p.get('name', '') for p in crew if isinstance(p, dict) and p.get('job') == 'Director'), '')

def get_list(items):
    return [p['name'] for p in items if isinstance(p, dict) and isinstance(p.get('name'), str)][:3]

def clean_data(value):
    if isinstance(value, list):
        return [''.join(item.lower().split()) for item in value]
    return ''.join(value.lower().split()) if isinstance(value, str) else ''

for feature in ['cast', 'crew', 'keywords', 'genres']:
    metadata[feature] = metadata[feature].apply(parse_list)
metadata['director'] = metadata['crew'].apply(get_director).apply(clean_data)
for feature in ['cast', 'keywords', 'genres']:
    metadata[feature] = metadata[feature].apply(get_list).apply(clean_data)

def create_soup(x):
    return ' '.join(x['keywords'] + x['cast'] + [x['director']] + x['genres'])

metadata['soup'] = metadata.apply(create_soup, axis=1)
display(metadata[['title', 'cast', 'director', 'keywords', 'genres', 'soup']].head(3))
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(metadata['soup'])
print('Matriz de metadatos:', count_matrix.shape)
display(get_recommendations('The Dark Knight Rises', count_matrix, metadata, tfidf_normalized=False))
display(get_recommendations('The Godfather', count_matrix, metadata, tfidf_normalized=False))

Modelo parcial: keywords y géneros; falta credits.csv para actores y director.
Matriz de metadatos: (45433, 9878)


,title,cast,director,keywords,genres,soup
0,Toy Story,[],,"[jealousy, toy, boy]","[animation, comedy, family]",jealousy toy boy animation comedy family
1,Jumanji,[],,"[boardgame, disappearance, basedonchildren'sbook]","[adventure, fantasy, family]",boardgame disappearance basedonchildren'sbook ...
2,Grumpier Old Men,[],,"[fishing, bestfriend, duringcreditsstinger]","[romance, comedy]",fishing bestfriend duringcreditsstinger roman...


,id,title,release_date,similarity
0,155,The Dark Knight,2008-07-16,0.833333
1,410921,Nothing to Lose,1994-01-01,0.707107
2,29185,An Innocent Man,1989-10-06,0.707107
3,25006,State Property,2002-01-18,0.707107
4,50380,Lockdown,2000-09-15,0.707107
5,45203,Once in the Life,2000-10-27,0.707107
6,96171,Men of Respect,1990-01-18,0.707107
7,25536,PTU,2003-04-17,0.707107
8,36221,Postman Blues,1997-08-16,0.707107
9,28943,Shiner,2000-09-22,0.707107


,id,title,release_date,similarity
0,195382,In the Name of the Law,1949-03-13,0.774597
1,259209,Feast of July,1995-10-12,0.632456
2,17961,L'Enfer,1994-02-16,0.632456
3,146341,Bird of Prey,1995-09-12,0.632456
4,12888,Belly,1998-11-04,0.632456
5,31863,True Believer,1989-02-17,0.632456
6,14290,Better Luck Tomorrow,2002-01-12,0.632456
7,44674,After Life,2002-09-12,0.632456
8,52677,The Crime of Monsieur Lange,1936-01-24,0.632456
9,63097,Golden Gate,1994-01-28,0.632456


## 4. Verificación y conclusiones
Comprobamos el orden del ranking, los límites de similitud y la exclusión de la película consultada. Estos controles verifican funcionamiento; la calidad de recomendaciones necesitaría evaluación con usuarios o interacciones reales.

El ranking reduce el efecto de calificaciones altas con pocos votos. Las sinopsis recuperan coincidencias temáticas; los metadatos añaden coincidencias de géneros y keywords y, al aportar `credits.csv`, también de reparto y director. No afirmamos que el modelo parcial reproduzca los resultados del modelo completo del curso.

Como siguientes experimentos se puede ordenar las 30 películas más similares por puntuación ponderada o aumentar el peso del director. No se implementa filtrado colaborativo porque faltan interacciones usuario-película.

In [5]:
assert q_movies['score'].is_monotonic_decreasing
assert q_movies['vote_count'].ge(m).all()
for title in ['The Dark Knight Rises', 'The Godfather']:
    for matrix, catalog, normalized in [(tfidf_matrix, df, True), (count_matrix, metadata, False)]:
        result = get_recommendations(title, matrix, catalog, tfidf_normalized=normalized)
        source_id = str(catalog.loc[catalog['title'].eq(title), 'id'].iloc[0])
        assert source_id not in result['id'].astype(str).tolist()
        assert len(result) <= 10
        assert result['similarity'].between(0, 1 + 1e-10).all()
        assert result['similarity'].is_monotonic_decreasing
try:
    get_recommendations('__pelicula_inexistente__')
except ValueError:
    pass
else:
    raise AssertionError('Debe detectar títulos inexistentes.')
print('Verificaciones completadas.')

Verificaciones completadas.
